# Analysis

Reads `df_evals.json`, computes every value quoted in the results section, and write the
two figures for our report. 

Conventions: A truncated trial has no answer (rather than a wrong answer), so
answer-level rates use only the answers the judge actually graded. And the 8 `all caps`
identification labels are grading artifacts, so they are set to False while the trials stay
in the table and keep every cell at n = 80.

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("future.no_silent_downcasting", True)

EVALS_PATH = Path("df_evals.json")
PLOTS = Path("plots") #where to save the plots
PLOTS.mkdir(exist_ok=True)

# Formatting for icml style
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "legend.fontsize": 6.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "0.9",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "legend.frameon": False,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})

BLUE, ORANGE, GREEN = "#0072B2", "#D55E00", "#009E73"
STRENGTHS = [1, 2, 4]
XPOS = np.arange(len(STRENGTHS))   # 1, 2, 4 are not equally spaced, so plot categorically

def save(fig, name): # always safe in pdf and png form
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS / f"{name}.{ext}")

## Load

In [ ]:
with open(EVALS_PATH) as f:
    df = pd.DataFrame(json.load(f))

ALL_CAPS_ARTIFACT = df["word"] == "all caps"

df["ident_reasoning"] = (df["identifying_reasoning"].fillna(False).astype(bool)
                         & ~ALL_CAPS_ARTIFACT)
df["ident_response"] = (df["identifying_response"].fillna(False).astype(bool)
                        & ~ALL_CAPS_ARTIFACT)

# The judge ran the identification prompt exactly when a detection was claimed.
df["claimed_in_reasoning"] = df["cot_enabled"] & df["identifying_reasoning"].notna()
df["claimed_in_response"] = df["identifying_response"].notna()

COND = ["prefilled", "cot_enabled", "strength"] # those are the conditions we group by for most of the analysis
no_think = df[df["prefilled"] & ~df["cot_enabled"]] 
pf_cot = df[df["prefilled"] & df["cot_enabled"]]

## Aggregates

In [ ]:
# Coherence and truncation
coherence_response = (df.assign(v=df["coherent_response"] == True)
                        .groupby(COND)["v"].mean().unstack("strength") * 100)
coherence_reasoning = (df[df["cot_enabled"]]
                         .assign(v=lambda d: d["coherent_reasoning"] == True)
                         .groupby(["prefilled", "strength"])["v"].mean()
                         .unstack("strength") * 100)
truncation = df.groupby(COND)["answer_truncated"].sum().unstack("strength")
truncation_by_concept = df[df["cot_enabled"]].groupby("word")["answer_truncated"].mean()

# The all caps labels
CAPS_TALK = r"all[ -]?caps|capital(?:s|ized|ization)|uppercase|shouting"
caps_hits = df[ALL_CAPS_ARTIFACT & (df["identifying_reasoning"] == True)]
caps_counts = pd.Series({
    "reasoning_labels": len(caps_hits),
    "response_labels": int((df["identifying_response"].fillna(False).astype(bool)
                            & ALL_CAPS_ARTIFACT).sum()),
    "naming_capitalisation": int(caps_hits["thinking"]
                                 .str.contains(CAPS_TALK, case=False).sum()),
    "bare_yes": int((caps_hits["identifying_reasoning_raw"].astype(str).str.strip()
                     == "YES").sum()),
})

# Identification at all three measurement points
identification = pd.DataFrame({
    "resp_hits":  no_think.groupby("strength")["ident_response"].sum(),
    "resp_n":     no_think.groupby("strength").size(),
    "trace_hits": pf_cot.groupby("strength")["ident_reasoning"].sum(),
    "trace_n":    pf_cot.groupby("strength")["claimed_in_reasoning"].sum(),
    "ans_hits":   pf_cot.groupby("strength")["ident_response"].sum(),
    "ans_n":      pf_cot.groupby("strength")["claimed_in_response"].sum(),
})
for k in ("resp", "trace", "ans"):
    identification[f"{k}_%"] = identification[f"{k}_hits"] / identification[f"{k}_n"] * 100

# Which concepts are ever identified, in any condition or metric.
concepts_identified = sorted(
    df.loc[df["ident_reasoning"] | df["ident_response"], "word"].unique())

# Unprefilled betrayal control at strength 2
ctrl = df[(~df["prefilled"]) & df["cot_enabled"] & (df["strength"] == 2)
          & (df["word"] == "betrayal")]
betrayal_control = pd.Series({
    "n": len(ctrl),
    "concept_in_trace": int(ctrl["thinking"].str.contains("betray", case=False).sum()),
    "coherent_traces": int((ctrl["coherent_reasoning"] == True).sum()),
    "truncated": int(ctrl["answer_truncated"].sum()),
    "graded_answers": int((~ctrl["answer_truncated"]).sum()),
    "answers_denying": int((ctrl["affirmative_response"] == False).sum()),
    "answers_identifying": int(ctrl["ident_response"].sum()),
})

# Prefilled betrayal at strength 2, where trace and answer counts diverge
b2 = df[df["prefilled"] & df["cot_enabled"] & (df["strength"] == 2)
        & (df["word"] == "betrayal")]
betrayal_thinking = pd.Series({
    "n": len(b2),
    "traces_identifying": int((b2["identifying_reasoning"] == True).sum()),
    "truncated": int(b2["answer_truncated"].sum()),
    "terminated": int((~b2["answer_truncated"]).sum()),
    "terminated_identifying": int((b2.loc[~b2["answer_truncated"],
                                          "identifying_response"] == True).sum()),
})

## Figure 1: coherence

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.75, 2.3), sharey=True)

styles = {
    (False, False): ("no prefill, no thinking", BLUE, "-", "o"),
    (False, True):  ("no prefill, thinking",    BLUE, "--", "s"),
    (True, False):  ("prefill, no thinking",    ORANGE, "-", "o"),
    (True, True):   ("prefill, thinking",       ORANGE, "--", "s"),
}
for key, (label, colour, ls, mk) in styles.items():
    ax1.plot(XPOS, coherence_response.loc[key].values,
             color=colour, ls=ls, marker=mk, ms=3, lw=1.2, label=label)

for pf, (label, colour) in {False: ("no prefill", BLUE), True: ("prefill", ORANGE)}.items():
    ax2.plot(XPOS, coherence_reasoning.loc[pf].values,
             color=colour, ls="--", marker="s", ms=3, lw=1.2, label=label)

ax1.set_title("(a) Final response")
ax2.set_title("(b) Reasoning trace (thinking on)")
ax1.set_ylabel("Coherent (%)")
for ax in (ax1, ax2):
    ax.set_xticks(XPOS)
    ax.set_xticklabels(STRENGTHS)
    ax.set_xlabel(r"Steering strength $\alpha$")
    ax.set_ylim(0, 105)
    ax.legend(loc="upper right")

save(fig, "31B_coherence")
plt.close(fig)

## Figure 2: identification

Prefilled cells only, all three measurement points in one figure. The denominators differ by
an order of magnitude, so every bar carries its raw fraction.

In [5]:
series = [
    ("resp",  "thinking off, response",       BLUE),
    ("trace", "thinking on, reasoning trace", ORANGE),
    ("ans",   "thinking on, final answer",    GREEN),
]

fig, ax = plt.subplots(figsize=(3.25, 2.3))
w = 0.26
for i, (key, label, colour) in enumerate(series):
    off = (i - 1) * w
    vals = identification[f"{key}_%"].values
    ax.bar(XPOS + off, vals, w * 0.92, color=colour, label=label)
    for x, v, hits, n in zip(XPOS, vals,
                             identification[f"{key}_hits"], identification[f"{key}_n"]):
        ax.text(x + off, v + 0.8, f"{hits}/{n}", ha="center", va="bottom",
                fontsize=5.5, rotation=90)

ax.set_xticks(XPOS)
ax.set_xticklabels(STRENGTHS)
ax.set_xlabel(r"Steering strength $\alpha$")
ax.set_ylabel("Correct identification (%)")
ax.set_ylim(0, 38)
ax.legend(loc="upper left", handlelength=1.0, borderaxespad=0.2)

save(fig, "31B_identification")
plt.close(fig)